# Train the Cited Newsroom Verifier SLM (Qwen3-1.7B + QLoRA)

This notebook runs the **training leg** of your project on a free Google Colab GPU.
Your Mac handles data-gen and eval; this notebook handles fine-tuning.

**What it does, top to bottom:**
1. Check you have a GPU.
2. Install Unsloth.
3. Upload your files (`eval.py`, your SFT training data, your golden eval set).
4. Load the base model `unsloth/Qwen3-1.7B`.
5. Generate **base** predictions (before training) — your baseline.
6. Add LoRA adapters + train (QLoRA).
7. Generate **tuned** predictions (after training).
8. Download the adapter + both prediction files.
9. Score `base` vs `tuned` back on your Mac with `eval.py`.

### Before you start
In the Colab menu: **Runtime -> Change runtime type -> T4 GPU -> Save.**
Then run each cell top to bottom with **Shift+Enter** (or Runtime -> Run all).

You will need these files from your Mac's `slm/` folder ready to upload:
- `eval.py`
- `data/train.sft.jsonl` — your 1,000 training examples
- `data/golden.json` — your 115-record eval golden set (kept out of training)

*(In Finder these are in `slm/` and `slm/data/`. The upload button flattens paths,
so just pick the files — they land in Colab's working dir where `from eval import`
and `load_testset("golden.json")` find them by name.)*

**Recommended first pass:** do a 2-minute smoke test with `junk.sft.jsonl` /
`junk.jsonl` (uncomment `max_steps` in Step 8) to confirm the whole pipeline runs
end-to-end before you commit ~30–45 min to the real 3-epoch run.

## Step 1 - Confirm you have a GPU

If the next cell errors with "command not found" or shows no GPU, you forgot to set
**Runtime -> Change runtime type -> T4 GPU**. Fix that, then re-run.

In [ ]:
!nvidia-smi

## Step 2 - Install Unsloth

This takes ~2-3 minutes. If you hit a weird dependency error, re-run this cell once
(Colab sometimes needs a second pass). It may ask you to restart the runtime the
first time - if so, click **Restart**, then continue from the next cell.

In [ ]:
%%capture
# Unsloth = the fast, low-memory QLoRA trainer. This is the whole reason a 1.7B
# model fits on a free T4 GPU. Unsloth pulls in matching transformers/trl/peft.
!pip install unsloth
# If you ever get a version-mismatch error, uncomment the force-reinstall below:
# !pip install --upgrade --force-reinstall --no-cache-dir unsloth unsloth_zoo

## Step 3 - Upload your files

Run the cell below. A **"Choose Files"** button appears - select these from your Mac:
- `eval.py`
- your SFT training file (`junk.sft.jsonl` for a smoke test, or `train.sft.jsonl` for real)
- your test set (`junk.jsonl` for a smoke test, or `testset.jsonl` for real)

Then set the two filename variables in the cell after it to match what you uploaded.

*(Uploaded files land in the current folder, so `from eval import ...` just works.)*

In [ ]:
from google.colab import files
uploaded = files.upload()  # pick eval.py + your .sft.jsonl + your testset .jsonl
print("\nUploaded:", list(uploaded.keys()))

In [ ]:
# ==== EDIT THESE TWO LINES to match the files you uploaded ====
SFT_FILE     = "train.sft.jsonl"  # your 1,000 training examples (chat format)
TESTSET_FILE = "golden.json"      # your 115-record eval golden set (base-vs-tuned)
# ==============================================================
# (For a fast smoke test first, upload + use "junk.sft.jsonl" / "junk.jsonl" instead.)

import os
for f in (SFT_FILE, TESTSET_FILE, "eval.py"):
    assert os.path.exists(f), f"Missing {f} - upload it in the cell above."
print("All required files present.")

## Step 4 - Load the base model

`unsloth/Qwen3-1.7B` in 4-bit. This is the exact model you'll fine-tune, loaded in a
memory-efficient form so it fits the free GPU.

In [ ]:
from unsloth import FastLanguageModel
import torch

# 3072 covers 100% of your training records (longest ~2.2k tokens); 2048 would
# silently TRUNCATE ~2% of them and corrupt their JSON training target.
MAX_SEQ_LEN = 3072

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen3-1.7B",
    max_seq_length = MAX_SEQ_LEN,
    load_in_4bit = True,     # QLoRA: 4-bit frozen base
    dtype = None,            # auto (bf16/fp16)
)
print("Base model loaded.")

## Step 5 - Baseline predictions (BEFORE training)

Critical: we capture how the **untrained** base model does on your test set *first*.
This is the "base" column of your base-vs-tuned table - the thing your fine-tune has
to beat. We reuse `build_messages` from your `eval.py` so the prompt format is
identical to training and to scoring. `enable_thinking=False` keeps the output as
clean JSON (no reasoning traces).

In [ ]:
import json
from eval import load_testset, build_messages

scenarios = load_testset(TESTSET_FILE)
print(f"Loaded {len(scenarios)} test scenarios.")

def generate_predictions(model, tokenizer, scenarios, out_path, max_new_tokens=512):
    """Run the model over each scenario, write eval-compatible predictions JSONL."""
    FastLanguageModel.for_inference(model)  # 2x faster inference mode
    with open(out_path, "w") as f:
        for i, scn in enumerate(scenarios, 1):
            messages = build_messages(scn)  # [system, user] from eval.py
            prompt = tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True,
                enable_thinking=False,  # Qwen3: no reasoning trace -> clean JSON
            )
            inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
            out = model.generate(
                **inputs, max_new_tokens=max_new_tokens, do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )
            text = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:],
                                    skip_special_tokens=True)
            f.write(json.dumps({"id": scn.id, "output": text}, ensure_ascii=False) + "\n")
            if i % 10 == 0 or i == len(scenarios):
                print(f"  {i}/{len(scenarios)}")
    print(f"Wrote {out_path}")

generate_predictions(model, tokenizer, scenarios, "base_preds.jsonl")

## Step 6 - Add LoRA adapters

We don't retrain the whole model - we attach small trainable "adapter" matrices
(LoRA) and train only those. `r` and `lora_alpha` are the main knobs; the defaults
below are the standard starting point. Remember the project rule: **fix problems in
your data, not by fiddling these numbers.**

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,                       # LoRA rank
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    use_gradient_checkpointing = "unsloth",  # saves memory
    random_state = 3407,
)
print("LoRA adapters attached. Trainable params:")
model.print_trainable_parameters()

## Step 7 - Prepare the training data

Your `*.sft.jsonl` has one record per line: `{"id", "messages":[system, user, assistant]}`.
We turn each into a single training string using Qwen3's chat template (thinking off),
so the model learns: given (system + user), produce the assistant's JSON verdicts.

In [ ]:
from datasets import Dataset

rows = [json.loads(l) for l in open(SFT_FILE)]
print(f"Loaded {len(rows)} training examples.")

def to_text(rec):
    return {
        "text": tokenizer.apply_chat_template(
            rec["messages"], tokenize=False, enable_thinking=False,
        )
    }

train_ds = Dataset.from_list(rows).map(to_text, remove_columns=["messages", "id"])
print("\nExample formatted training text:\n")
print(train_ds[0]["text"][:1200])

## Step 8 - Train

`train_on_responses_only` makes the model learn only from the assistant's JSON (it
doesn't waste capacity memorizing your prompts). For a first/smoke run this takes a
few minutes. With real data, bump `num_train_epochs` to 2-3 and remove `max_steps`.

Watch the **loss** printout drop - that's the model learning. A curve falling from
~1.5-2.0 toward ~0.5-1.0 is healthy.

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_ds,
    args = SFTConfig(
        dataset_text_field = "text",
        max_seq_length = MAX_SEQ_LEN,
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,   # effective batch size = 8
        warmup_steps = 5,
        num_train_epochs = 3,              # real run: full 3 epochs over 1,000 examples
        # max_steps = 60,                  # (smoke-test cap — leave commented for the real run)
        learning_rate = 2e-4,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
    ),
)

# Train only on the assistant's response (Qwen3 chat markers).
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",
    response_part = "<|im_start|>assistant\n",
)

trainer_stats = trainer.train()
print("Done training.")

## Step 9 - Tuned predictions (AFTER training)

Same test set, same prompts - now with the trained adapter. This is the "tuned"
column of your table.

In [ ]:
generate_predictions(model, tokenizer, scenarios, "tuned_preds.jsonl")

## Step 10 - Quick score right here (optional)

`eval.py`'s deterministic metrics are pure Python, so you can see base-vs-tuned
numbers immediately in Colab. (The LLM-as-judge part needs your API key and is
better run on your Mac.) This is your **midweek gate**: watch `knowledge_leakage_rate`
and `fabricated_citation_rate` drop from base to tuned.

In [ ]:
# In Colab, {TESTSET_FILE} injects the Python variable into the shell command.
!python eval.py score --testset {TESTSET_FILE} --base base_preds.jsonl --tuned tuned_preds.jsonl --out results.md

## Step 11 - Save the adapter and download everything

Saves the trained LoRA adapter (tiny - a few MB), zips it, and downloads it along
with `base_preds.jsonl`, `tuned_preds.jsonl`, and `results.md`.

Back on your Mac you can re-run the full eval (including the LLM-as-judge):
```bash
python3 eval.py score --testset data/testset.jsonl \
    --base base_preds.jsonl --tuned tuned_preds.jsonl --judge --out results.md
```

**Colab wipes everything when the session ends - so actually download the files.**

In [ ]:
ADAPTER_DIR = "qwen3-verifier-lora"
model.save_pretrained(ADAPTER_DIR)      # LoRA adapter only (small)
tokenizer.save_pretrained(ADAPTER_DIR)
!zip -r -q {ADAPTER_DIR}.zip {ADAPTER_DIR}
print("Saved + zipped adapter.")

from google.colab import files
for fname in [f"{ADAPTER_DIR}.zip", "base_preds.jsonl", "tuned_preds.jsonl", "results.md"]:
    try:
        files.download(fname)
    except Exception as e:
        print(f"(skip {fname}: {e})")

## Appendix (later in the week) - Publish to Hugging Face Hub

Your final submission needs the model on the HF Hub. When you're ready, get a token
from huggingface.co/settings/tokens (write access), then run the cell below. Skip
this for now - it's a Day 5 task, not needed to train.

In [ ]:
# from huggingface_hub import login
# login()  # paste your write token when prompted
# HF_REPO = "your-username/qwen3-1.7b-newsroom-verifier"
# model.push_to_hub(HF_REPO)
# tokenizer.push_to_hub(HF_REPO)
# print("Pushed to", HF_REPO)